# Medical Code Intelligence — Full Pipeline Demo

This notebook demonstrates every component of the Medical Code Intelligence pipeline:

| # | Component | Module | GPU? |
|---|-----------|--------|------|
| 1 | Configuration | `configs.ner_config` | No |
| 2 | Shorthand Expansion | `src.clinical.shorthand` | No |
| 3 | Negation Detection (rule-based) | `src.clinical.negation` | No |
| 4 | ICD-10-CM Code Lookup | `src.clinical.icd_codes` | No |
| 5 | MS-DRG Cost Estimation | `src.clinical.drg_costs` | No |
| 6 | Entity Post-Processing | `src.inference.entity_utils` | No |
| 7 | Evaluation Metrics | `src.evaluation.metrics` | No |
| 8 | Curated ICD Dataset Generation | `src.data.icd_dataset` | No |
| 9 | MedMentions & MACCROBAT (optional sources) | `src.data.icd_dataset` | No* |
| 10 | End-to-End Pipeline (pre-extracted entities) | `src.clinical.pipeline` | No |
| 11 | Adversarial Training (overview) | `src.training.adversarial` | No** |
| 12 | Assertion Classifier (transformer) | `src.clinical.assertion` | Optional |
| **13** | **Dataset Loading & Preprocessing** | `src.data.dataset_loader` | No |
| **14** | **Model Building & Training** | `src.training.trainer` | GPU rec. |
| **15** | **Evaluation & Error Analysis** | `src.evaluation` | No |
| **16** | **Full Pipeline — Shorthand to Cost Estimate** | `src.clinical.pipeline` | GPU rec. |
| **17** | **Multi-Model Training on ICD Dataset** | `src.data.icd_dataset` | GPU rec. |
| **18** | **ICD-Trained Pipeline — Shorthand, Negation & DRG Cost** | `src.clinical.pipeline` | GPU rec. |
| 19 | CLI Scripts Reference | — | — |
| 20 | Running the Test Suite | — | — |

\* MedMentions and MACCROBAT download from HuggingFace on first use; cells show the loader API and structure with graceful fallback if unavailable.

\** Adversarial training overview runs on CPU with a toy model. Actual adversarial training requires a GPU.

Sections 13-18 train real NER models and run the complete six-stage pipeline (shorthand expansion → NER → negation → ICD coding → DRG cost estimation) on clinical notes. Section 17 trains multiple biomedical models on the full 7-source ICD composite dataset. Section 18 uses the best model to run the full pipeline on three clinical cases with step-by-step stage output and DRG cost analysis.

In [ ]:
!git clone https://github.com/jcl347/Medical_Code_Intelligence
%cd Medical_Code_Intelligence
!pip install -r requirements.txt -q

In [ ]:
import os, sys, pathlib

# After %cd Medical_Code_Intelligence, cwd is the repo root.
# This cell also handles running from notebooks/ or other locations.
def _find_repo_root():
    markers = ("src", "configs")

    def _has_markers(p):
        return all((p / m).is_dir() for m in markers)

    cwd = pathlib.Path.cwd()

    # 1. cwd IS the repo root
    if _has_markers(cwd):
        return str(cwd)

    # 2. cwd is notebooks/ inside the repo
    if _has_markers(cwd.parent):
        return str(cwd.parent)

    # 3. Repo is a subdirectory of cwd (Colab default: /content)
    for child in sorted(cwd.iterdir()):
        if child.is_dir() and _has_markers(child):
            return str(child)

    # 4. Walk up from cwd
    p = cwd
    for _ in range(5):
        p = p.parent
        if _has_markers(p):
            return str(p)

    raise RuntimeError(
        f"Cannot find repo root (looked for src/ + configs/ dirs).\n"
        f"  cwd = {cwd}\n"
        f"Hint: run the setup cell above, or set manually:\n"
        f"  REPO_ROOT = '/path/to/Medical_Code_Intelligence'"
    )

REPO_ROOT = _find_repo_root()
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
os.chdir(REPO_ROOT)

print(f"Repo root: {REPO_ROOT}")
print(f"  configs/ exists: {os.path.isdir(os.path.join(REPO_ROOT, 'configs'))}")
print(f"  src/ exists:     {os.path.isdir(os.path.join(REPO_ROOT, 'src'))}")

---
## 1. Configuration — `NERConfig`, `MODEL_CONFIGS`, `DATASET_CONFIGS`

All training hyperparameters, model definitions, and dataset metadata live in a single dataclass.

In [ ]:
from configs.ner_config import NERConfig, MODEL_CONFIGS, DATASET_CONFIGS

# --- Inspect available models ---
print("=== Supported Pre-trained Models ===")
for key, cfg in MODEL_CONFIGS.items():
    print(f"  {key:20s}  {cfg['model_name']}")

print()

# --- Inspect available datasets ---
print("=== Supported Datasets ===")
for key, cfg in DATASET_CONFIGS.items():
    print(f"  {key:20s}  {cfg['description'][:60]}")

In [ ]:
# --- Default hyperparameters ---
config = NERConfig()
print("=== Default NERConfig ===")
for k, v in vars(config).items():
    print(f"  {k:35s} = {v}")

In [ ]:
# --- Override for a specific experiment ---
custom = NERConfig(
    model_key="bio_clinicalbert",
    dataset_key="icd_ner",
    learning_rate=3e-5,
    num_train_epochs=15,
    use_adversarial_training=True,
    adv_method="fgm",
    resolve_drg=True,
)
print(f"Model:       {custom.model_key}")
print(f"Dataset:     {custom.dataset_key}")
print(f"LR:          {custom.learning_rate}")
print(f"Adversarial: {custom.adv_method} (epsilon={custom.adv_epsilon})")
print(f"DRG enabled: {custom.resolve_drg}")

---
## 2. Shorthand Expansion — `ShorthandExpander`

Expands physician abbreviations ("cp" → "chest pain") with character offset tracking for NER alignment.

Uses the built-in fallback (~280 abbreviations) so no network download is required.

In [ ]:
from src.clinical.shorthand import ShorthandExpander

expander = ShorthandExpander(source="builtin")
print(f"Loaded {expander.num_abbreviations} abbreviations")
print(f"Ambiguous: {expander.num_ambiguous}")

In [ ]:
# --- Simple expansion ---
samples = [
    "pt c/o sob and cp",
    "hx of dm2, htn, and cad",
    "dx: afib r/o mi",
    "nkda, aox3, wnl",
]

print("=== Shorthand Expansion ===")
for text in samples:
    expanded = expander.expand(text)
    print(f"  {text:35s} → {expanded}")

In [ ]:
# --- Expansion with offset tracking ---
text = "pt denies cp or sob"
expanded, offsets = expander.expand_with_offsets(text)
print(f"Original:  {text!r}")
print(f"Expanded:  {expanded!r}")
print(f"\nOffset map ({len(offsets)} expansions):")
for om in offsets:
    print(f"  '{om['abbreviation']}' @ [{om['original_start']}:{om['original_end']}] "
          f"→ '{om['expansion']}' @ [{om['expanded_start']}:{om['expanded_end']}]")

In [ ]:
# --- Identify abbreviations without expanding ---
abbrevs = expander.identify_abbreviations("pt c/o sob and cp on exertion")
print("=== Identified Abbreviations ===")
for a in abbrevs:
    print(f"  '{a['abbreviation']}' @ [{a['start']}:{a['end']}] → '{a['expansion']}'")

---
## 3. Negation Detection — `NegationDetector`

Rule-based ConText/NegEx algorithm with 100+ trigger patterns. Detects six assertion statuses:
**AFFIRMED**, **NEGATED**, **POSSIBLE**, **HYPOTHETICAL**, **HISTORICAL**, **FAMILY**.

In [ ]:
from src.clinical.negation import NegationDetector, NegationStatus

detector = NegationDetector(scope_window=6)

# --- Show all assertion statuses ---
print("Assertion statuses:", [s.value for s in NegationStatus])

In [ ]:
# --- Detect negation scopes in raw text ---
text = "Patient denies chest pain but has persistent cough. No fever. History of diabetes."
scopes = detector.detect(text)
print(f"Text: {text!r}\n")
print(f"Detected {len(scopes)} negation/context scopes:")
for s in scopes:
    print(f"  [{s.status.value:12s}] trigger='{s.trigger_text}' "
          f"scope=[{s.scope_start}:{s.scope_end}] → '{text[s.scope_start:s.scope_end]}' "
          f"({s.direction})")

In [ ]:
# --- Annotate pre-extracted entities ---
text = "Patient denies fever but reports persistent cough. No evidence of pneumonia. Family history of diabetes."
entities = [
    {"text": "fever",     "label": "DIAGNOSIS", "start": 15, "end": 20},
    {"text": "cough",     "label": "DIAGNOSIS", "start": 43, "end": 48},
    {"text": "pneumonia", "label": "DIAGNOSIS", "start": 67, "end": 76},
    {"text": "diabetes",  "label": "DIAGNOSIS", "start": 96, "end": 104},
]

annotated = detector.annotate_entities(text, entities)
print(f"Text: {text!r}\n")
print("Entity Annotations:")
for ent in annotated:
    trigger = ent.get('negation_trigger', '-')
    print(f"  {ent['text']:15s} → {ent['negation']:12s} (trigger: {trigger})")

In [ ]:
# --- Quick negation check ---
text = "No evidence of pulmonary embolism."
print(f"'{text}' — is 'pulmonary embolism' negated? "
      f"{detector.is_negated(text, 15, 33)}")

text2 = "Diagnosed with pulmonary embolism."
print(f"'{text2}' — is 'pulmonary embolism' negated? "
      f"{detector.is_negated(text2, 16, 34)}")

In [ ]:
# --- Test all six assertion statuses ---
test_cases = [
    ("Patient has pneumonia.", "pneumonia", 12, 21, "affirmed"),
    ("Patient denies chest pain.", "chest pain", 15, 25, "negated"),
    ("Possible diagnosis of lupus.", "lupus", 23, 28, "possible"),
    ("If symptoms worsen, consider asthma.", "asthma", 30, 36, "hypothetical"),
    ("History of myocardial infarction.", "myocardial infarction", 11, 32, "historical"),
    ("Family history of breast cancer.", "breast cancer", 18, 31, "family"),
]

print("=== All Six Assertion Statuses ===")
for text, entity, start, end, expected in test_cases:
    ents = [{"text": entity, "label": "DIAGNOSIS", "start": start, "end": end}]
    result = detector.annotate_entities(text, ents)
    status = result[0]["negation"]
    match = "✓" if status == expected else "✗"
    print(f"  {match} {status:12s} (expected {expected:12s}) — {text}")

---
## 4. ICD-10-CM Code Lookup — `ICDCodeLookup`

TF-IDF character n-gram matching against 51K ICD-10-CM codes (falls back to 45 built-in codes offline).

In [ ]:
from src.clinical.icd_codes import ICDCodeLookup

lookup = ICDCodeLookup()
print(f"Loaded {len(lookup._codes)} ICD-10-CM codes")

In [ ]:
# --- Match entity text to ICD codes ---
queries = [
    "chest pain",
    "type 2 diabetes mellitus",
    "hypertension",
    "congestive heart failure",
    "pneumonia",
    "atrial fibrillation",
    "chronic kidney disease",
]

print("=== ICD-10-CM Entity Linking ===")
for query in queries:
    matches = lookup.match_entity(query, top_k=3)
    top = matches[0] if matches else None
    if top:
        print(f"  {query:30s} → {top.code}: {top.description} (score={top.score:.3f})")
    else:
        print(f"  {query:30s} → no match")

In [ ]:
# --- Direct code lookup ---
codes_to_look_up = ["E11.9", "I10", "J18.9", "R07.9", "I50.9"]

print("=== Direct Code Lookup ===")
for code_str in codes_to_look_up:
    code_obj = lookup.lookup_code(code_str)
    if code_obj:
        print(f"  {code_obj.code}: {code_obj.description}")
    else:
        print(f"  {code_str}: not found")

In [ ]:
# --- Batch entity matching ---
batch_entities = [
    {"text": "hypertension", "label": "DIAGNOSIS"},
    {"text": "pneumonia", "label": "DIAGNOSIS"},
    {"text": "chest pain", "label": "DIAGNOSIS"},
    {"text": "diabetes", "label": "DIAGNOSIS"},
]

results = lookup.match_entities_batch(batch_entities, top_k=3)
print("=== Batch Entity → ICD Mapping ===")
for r in results:
    codes = [c["code"] for c in r.get("icd_codes", [])]
    print(f"  {r['text']:20s} → {codes}")

---
## 5. MS-DRG Cost Estimation — `DRGCostEstimator`

Maps ICD-10-CM codes to MS-DRGs and estimates financial impact using **drgpy** for ICD→DRG grouping and DRG metadata, with **real CMS FY 2026 relative weights** auto-downloaded from NBER.

> **Data sources:**
> - `drgpy` (Apache 2.0) provides the ICD→DRG grouper and complete DRG catalog (767 DRGs with titles, MDC, type).
> - **NBER CMS Table 5 CSV** — official FY 2026 relative weights, geometric and arithmetic mean LOS for ~770 DRGs, auto-downloaded from `data.nber.org` on first use and cached locally.
> - Local CMS IPPS Table 5 Excel files can also be provided via `table5_path` for custom overlays.

In [ ]:
from src.clinical.drg_costs import DRGCostEstimator, DRGResult, CostImpactAnalysis

estimator = DRGCostEstimator()
print(f"Base rate: ${estimator.base_rate:,.2f} (FY 2026)")
print(f"DRG catalog: {estimator.num_drgs} DRGs loaded (drgpy metadata + NBER CMS Table 5 weights)")
print(f"Grouper available: {estimator._grouper is not None}")

In [ ]:
# --- Direct cost estimate by DRG code ---
drg_codes = ["291", "292", "293", "065", "066", "067", "189", "190", "191"]

print("=== DRG Cost Estimates ===")
for code in drg_codes:
    result = estimator._build_result(code)
    if result:
        print(f"  DRG {result.drg_code}: {result.drg_title:50s} "
              f"wt={result.relative_weight:.4f}  ${result.estimated_payment:>10,.2f}  [{result.severity_level}]")

In [ ]:
# --- DRG grouping from ICD codes ---
# drgpy provides full ICD→DRG grouping for all MS-DRGs with CC/MCC evaluation.
# Real CMS FY 2026 relative weights are auto-loaded from NBER Table 5.
icd_sets = [
    (["J18.9"],                       "Pneumonia alone"),
    (["J18.9", "E11.9"],              "Pneumonia + diabetes (no CC)"),
    (["J18.9", "E11.9", "N17.9"],     "Pneumonia + diabetes + AKI (CC!)"),
    (["I50.9"],                       "Heart failure alone"),
    (["I50.9", "E11.9", "N17.9"],     "Heart failure + diabetes + AKI"),
    (["A41.9"],                       "Sepsis alone"),
    (["P39.3", "I50.89", "N18.9"],    "Neonatal UTI + HF + CKD"),
    (["I21.9", "E11.9"],              "Acute MI + diabetes"),
    (["G40.909"],                     "Epilepsy / seizures"),
    (["K70.30"],                      "Alcoholic cirrhosis"),
]

print(f"=== ICD → DRG Grouping ({estimator.num_drgs} DRGs, CMS FY 2026 weights) ===\n")
for codes, desc in icd_sets:
    result = estimator.get_drg(codes)
    if result:
        print(f"  {str(codes):45s} → DRG {result.drg_code}: {result.drg_title[:50]}")
        print(f"    {desc} — wt={result.relative_weight:.4f}, ${result.estimated_payment:,.2f}")
    else:
        print(f"  {str(codes):45s} → (ungroupable)")
        print(f"    {desc}")

print("\nNote: Adding AKI (N17.9) as a secondary diagnosis acts as a CC,")
print("bumping Pneumonia from DRG 195 (base) to 194 (with CC).")

In [ ]:
# --- Cost impact analysis (CC/MCC comparison) ---
# Using real CMS FY 2026 relative weights from NBER Table 5.

# Heart Failure family: DRG 291 (MCC) / 292 (CC) / 293 (base)
print("=== Heart Failure DRG Family (291/292/293) — CMS FY 2026 Weights ===")
for code in ["291", "292", "293"]:
    r = estimator._build_result(code)
    if r:
        print(f"  DRG {r.drg_code} [{r.severity_level:4s}]: wt={r.relative_weight:.4f}  "
              f"${r.estimated_payment:>10,.2f}  LOS={r.geometric_mean_los:.1f}d  {r.drg_title}")

# Calculate revenue at risk
base = estimator._build_result("293")
mcc = estimator._build_result("291")
if base and mcc:
    gap = mcc.estimated_payment - base.estimated_payment
    print(f"\n  Revenue at risk (base→MCC): ${gap:,.2f}")

In [ ]:
# --- Full cost impact analysis via API ---
# analyze_cost_impact() uses drgpy for ICD→DRG grouping + NBER CMS weights.
# analyze_drg_family() works with a known DRG code directly (no grouping needed).
analysis = estimator.analyze_cost_impact(["J18.9", "E11.9"])
if analysis is None:
    # Fallback: analyse a known DRG directly from the weight table
    print("(drgpy not installed — using analyze_drg_family with known DRG code)\n")
    analysis = estimator.analyze_drg_family("292")   # Heart Failure w CC

if analysis:
    print("=== Cost Impact Analysis (CMS FY 2026 Weights) ===")
    d = analysis.to_dict()
    print(f"  Current DRG: {d['current']['drg_code']} — {d['current']['drg_title']}")
    print(f"  Relative weight: {d['current']['relative_weight']:.4f}")
    print(f"  Estimated payment: ${d['current']['estimated_payment']:,.2f}")
    print(f"  Revenue at risk: ${d['revenue_at_risk']:,.2f}")
    print(f"  Undercoding risk: {d['undercoding_risk']}")
    if 'mcc_variant' in d:
        print(f"  MCC variant: DRG {d['mcc_variant']['drg_code']} — "
              f"wt={d['mcc_variant']['relative_weight']:.4f}, "
              f"${d['mcc_variant']['estimated_payment']:,.2f}")
else:
    print("No analysis available (DRG code not in weight table).")

---
## 6. Entity Post-Processing — `post_process_entities()`

Filters garbage entities (stopwords, punctuation) and merges adjacent fragments from subword tokenization.

In [ ]:
from src.inference.entity_utils import NEREntity, post_process_entities

text = "Patient has congestive heart failure and type 2 diabetes mellitus."

# Simulate raw NER output with garbage and fragments
raw_entities = [
    NEREntity(text="congestive",    label="DIAGNOSIS", start_char=12, end_char=22, score=0.95),
    NEREntity(text="heart failure", label="DIAGNOSIS", start_char=23, end_char=36, score=0.93),
    NEREntity(text="and",           label="DIAGNOSIS", start_char=37, end_char=40, score=0.30),
    NEREntity(text="type",          label="DIAGNOSIS", start_char=41, end_char=45, score=0.25),
    NEREntity(text="2 diabetes mellitus", label="DIAGNOSIS", start_char=46, end_char=65, score=0.91),
]

print(f"Before post-processing ({len(raw_entities)} entities):")
for e in raw_entities:
    print(f"  '{e.text}' [{e.label}] score={e.score:.2f}")

cleaned = post_process_entities(raw_entities, text)
print(f"\nAfter post-processing ({len(cleaned)} entities):")
for e in cleaned:
    print(f"  '{e.text}' [{e.label}] score={e.score:.2f}")

---
## 7. Evaluation Metrics — `compute_ner_metrics()`

Entity-level precision, recall, and F1 using seqeval (or built-in fallback).

In [ ]:
from src.evaluation.metrics import compute_ner_metrics, _extract_entities_from_bio
import numpy as np

# --- Simulated model predictions ---
# Label mapping: 0=O, 1=B-DIAGNOSIS, 2=I-DIAGNOSIS
label_list = ["O", "B-DIAGNOSIS", "I-DIAGNOSIS"]

# Gold:  "The patient has [congestive heart failure] and [diabetes]."
# Pred:  "The patient has [congestive heart] failure and [diabetes]."
#  (boundary error on first entity, correct on second)

gold_labels = [0, 0, 0, 1, 2, 2, 0, 1, 0]  # O O O B I I O B O
pred_labels = [0, 0, 0, 1, 2, 0, 0, 1, 0]  # O O O B I O O B O (missed I on "failure")

# compute_ner_metrics expects (predictions, labels, label_list) as separate arrays
predictions = np.array([pred_labels])
labels = np.array([gold_labels])

metrics = compute_ner_metrics(predictions, labels, label_list=label_list)

print("=== Entity-Level Metrics ===")
for k, v in metrics.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.4f}")
    else:
        print(f"  {k}:")
        print(v)

print("\n(Note: boundary error on 'congestive heart failure' causes lower recall)")

In [ ]:
# --- BIO entity extraction utility ---
labels = ["O", "B-DIAGNOSIS", "I-DIAGNOSIS", "I-DIAGNOSIS", "O", "B-DIAGNOSIS", "O"]
entities = _extract_entities_from_bio(labels)
print("Extracted entities from BIO sequence:")
for etype, start, end in sorted(entities):
    print(f"  {etype} @ tokens [{start}:{end}]")

---
## 8. Curated ICD Dataset — Template-Generated Examples

The `icd_ner` dataset includes ~100 template-generated sentences targeting common NER failure patterns.

In [ ]:
from src.data.icd_dataset import _generate_template_examples, ICD_NER_LABELS

print(f"Label scheme: {ICD_NER_LABELS}\n")

examples = _generate_template_examples()
print(f"Generated {len(examples)} template examples\n")

# Show a few examples
print("=== Sample Template Examples ===")
for ex in examples[:8]:
    tokens = ex["tokens"]
    labels = ex["labels"]
    # Reconstruct text with labels
    labeled = []
    for tok, lab in zip(tokens, labels):
        if lab.startswith("B-"):
            labeled.append(f"[{tok}")
        elif lab.startswith("I-"):
            labeled.append(tok)
        else:
            if labeled and labeled[-1] and not labeled[-1].endswith("]"):
                # Close the previous entity bracket
                labeled[-1] = labeled[-1] + "]"
            labeled.append(tok)
    # Close any trailing entity
    text = " ".join(labeled)
    if text.count("[") > text.count("]"):
        text += "]"
    print(f"  {text}")

---
## 9. MedMentions & MACCROBAT — Optional Dataset Sources

The `icd_ner` composite dataset includes two optional HuggingFace sources that download on first use:

1. **MedMentions** (`bigbio/medmentions`) — up to 5K examples from 4,392 PubMed abstracts with 350K+ UMLS entity mentions, filtered for disease/disorder semantic types (T047, T048, T019, T046, T191)
2. **MACCROBAT** (`singh-aditya/MACCROBAT_biomedical_ner`) — up to 3K examples from 200 clinical case reports with DISEASE_DISORDER entities, providing clinical-note-style text that PubMed abstracts lack

Both are loaded by `load_icd_ner_dataset()` as Sources 6 and 7. If the download fails, they are skipped gracefully.

In [ ]:
# --- Source 6: MedMentions ---
# Loads disease/disorder entities from 4,392 PubMed abstracts (bigbio/medmentions).
# Filters for UMLS semantic types: T047 (Disease), T048 (Mental Disorder),
# T019 (Congenital Abnormality), T046 (Pathologic Function), T191 (Neoplastic Process).

from src.data.icd_dataset import _load_medmentions_diseases, _MEDMENTIONS_DISEASE_TYPES

print("=== MedMentions Disease Loader ===")
print(f"Target UMLS semantic types: {sorted(_MEDMENTIONS_DISEASE_TYPES)}")
print()

try:
    mm_dataset = _load_medmentions_diseases(max_examples=50)  # small sample for demo
    for split, ds in mm_dataset.items():
        n_entities = sum(1 for ex in ds for lab in ex["ner_labels"] if lab.startswith("B-"))
        print(f"  {split:12s}: {len(ds):4d} examples, {n_entities} DIAGNOSIS entities")

    # Show a few examples
    print("\nSample MedMentions examples:")
    for ex in list(mm_dataset["train"])[:3]:
        tokens = ex["tokens"]
        labels = ex["ner_labels"]
        # Show only the diagnosis spans
        spans = []
        current = []
        for tok, lab in zip(tokens, labels):
            if lab.startswith("B-"):
                if current:
                    spans.append(" ".join(current))
                current = [tok]
            elif lab.startswith("I-") and current:
                current.append(tok)
            else:
                if current:
                    spans.append(" ".join(current))
                    current = []
        if current:
            spans.append(" ".join(current))
        text_preview = " ".join(tokens[:15])
        if len(tokens) > 15:
            text_preview += " ..."
        print(f"  Text: {text_preview}")
        print(f"  Entities: {spans}")
        print()
except Exception as e:
    print(f"  MedMentions not available (expected in offline mode): {type(e).__name__}: {e}")
    print("  This source is optional — load_icd_ner_dataset() skips it gracefully.")

In [ ]:
# --- Source 7: MACCROBAT ---
# Loads DISEASE_DISORDER entities from 200 clinical case reports
# (singh-aditya/MACCROBAT_biomedical_ner). Provides clinical-note-style text
# that PubMed abstracts lack, closing the domain gap.

from src.data.icd_dataset import _load_maccrobat_diseases, _MACCROBAT_DISEASE_LABELS

print("=== MACCROBAT Disease Loader ===")
print(f"Target entity labels: {sorted(_MACCROBAT_DISEASE_LABELS)}")
print()

try:
    mac_dataset = _load_maccrobat_diseases(max_examples=50)  # small sample for demo
    for split, ds in mac_dataset.items():
        n_entities = sum(1 for ex in ds for lab in ex["ner_labels"] if lab.startswith("B-"))
        print(f"  {split:12s}: {len(ds):4d} examples, {n_entities} DIAGNOSIS entities")

    # Show a few examples
    print("\nSample MACCROBAT examples:")
    for ex in list(mac_dataset["train"])[:3]:
        tokens = ex["tokens"]
        labels = ex["ner_labels"]
        # Show only the diagnosis spans
        spans = []
        current = []
        for tok, lab in zip(tokens, labels):
            if lab.startswith("B-"):
                if current:
                    spans.append(" ".join(current))
                current = [tok]
            elif lab.startswith("I-") and current:
                current.append(tok)
            else:
                if current:
                    spans.append(" ".join(current))
                    current = []
        if current:
            spans.append(" ".join(current))
        text_preview = " ".join(tokens[:15])
        if len(tokens) > 15:
            text_preview += " ..."
        print(f"  Text: {text_preview}")
        print(f"  Entities: {spans}")
        print()
except Exception as e:
    print(f"  MACCROBAT not available (expected in offline mode): {type(e).__name__}: {e}")
    print("  This source is optional — load_icd_ner_dataset() skips it gracefully.")

---
## 10. End-to-End Pipeline — `MedicalCodingPipeline`

Chains shorthand expansion → negation detection → ICD resolution → DRG cost estimation.

Using `process_with_entities()` to supply pre-extracted entities (no NER model required).

In [ ]:
from src.clinical.pipeline import MedicalCodingPipeline, MedicalEntity

# Initialize pipeline without a trained NER model
pipeline = MedicalCodingPipeline(
    model_path=None,         # No NER model — we'll supply entities manually
    expand_shorthand=True,
    detect_negation=True,
    negation_strategy="rules",
    resolve_icd_codes=True,
    icd_top_k=3,
    resolve_drg=False,       # Set True to enable DRG cost estimation (NBER Table 5 auto-loaded)
)

print("Pipeline initialized:")
print(f"  Shorthand expander: {pipeline.shorthand_expander is not None}")
print(f"  Negation detector:  {pipeline.negation_detector is not None}")
print(f"  ICD lookup:         {pipeline.icd_lookup is not None}")
print(f"  DRG estimator:      {pipeline.drg_estimator is not None}")

In [ ]:
# --- Process pre-extracted entities ---
clinical_text = "Patient denies chest pain. Diagnosed with congestive heart failure and hypertension."

pre_extracted = [
    {"text": "chest pain",              "label": "DIAGNOSIS", "start": 15, "end": 25, "score": 0.95},
    {"text": "congestive heart failure", "label": "DIAGNOSIS", "start": 43, "end": 66, "score": 0.97},
    {"text": "hypertension",            "label": "DIAGNOSIS", "start": 71, "end": 83, "score": 0.96},
]

results = pipeline.process_with_entities(clinical_text, pre_extracted)

print(f"Input:  {clinical_text}\n")
print("=== Pipeline Results ===")
for ent in results:
    print(f"  Entity: {ent.text}")
    print(f"    Label:    {ent.label}")
    print(f"    Negation: {ent.negation} (trigger: {ent.negation_trigger or 'none'})")
    print(f"    Score:    {ent.score:.3f}")
    if ent.icd_codes:
        print(f"    ICD codes:")
        for icd in ent.icd_codes[:2]:
            print(f"      {icd['code']}: {icd['description']} (score={icd['score']:.3f})")
    print()

In [ ]:
# --- Human-readable formatted output ---
formatted = pipeline.format_output(clinical_text, results)
print(formatted)

In [ ]:
# --- MedicalEntity properties and serialization ---
for ent in results:
    print(f"  {ent.text:30s} is_affirmed={ent.is_affirmed}  is_negated={ent.is_negated}")

print("\n=== JSON serialization ===")
import json, numpy as np

class NumpyEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, (np.floating, np.integer)):
            return float(obj)
        return super().default(obj)

print(json.dumps(results[0].to_dict(), indent=2, cls=NumpyEncoder))

In [ ]:
# --- Multiple clinical scenarios ---
scenarios = [
    (
        "No evidence of pneumonia on chest X-ray. Patient has COPD exacerbation.",
        [
            {"text": "pneumonia",         "label": "DIAGNOSIS", "start": 15, "end": 24, "score": 0.92},
            {"text": "COPD exacerbation", "label": "DIAGNOSIS", "start": 43, "end": 60, "score": 0.94},
        ]
    ),
    (
        "History of stroke. Currently presents with acute kidney injury.",
        [
            {"text": "stroke",             "label": "DIAGNOSIS", "start": 11, "end": 17, "score": 0.90},
            {"text": "acute kidney injury", "label": "DIAGNOSIS", "start": 43, "end": 61, "score": 0.96},
        ]
    ),
    (
        "Mother had breast cancer. Patient denies any malignancy.",
        [
            {"text": "breast cancer", "label": "DIAGNOSIS", "start": 11, "end": 24, "score": 0.93},
            {"text": "malignancy",   "label": "DIAGNOSIS", "start": 45, "end": 55, "score": 0.88},
        ]
    ),
]

print("=== Multiple Clinical Scenarios ===")
for text, ents in scenarios:
    results = pipeline.process_with_entities(text, ents)
    print(f"\n{text}")
    for r in results:
        icd = r.icd_codes[0]['code'] if r.icd_codes else 'N/A'
        print(f"  [{r.text}] {r.negation.upper():12s} ICD={icd}")

---
## 11. Adversarial Training — `FGM`, `PGD`, `AdversarialTrainer`

FGM and PGD perturb word embeddings during training to improve robustness (+0.5-1.5% F1).

This section demonstrates the API structure. Actual training requires a GPU and dataset.

In [ ]:
import torch
from src.training.adversarial import FGM, PGD, AdversarialTrainer

# --- Demonstrate FGM on a toy model ---
class ToyModel(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.word_embeddings = torch.nn.Embedding(100, 16)
        self.classifier = torch.nn.Linear(16, 3)

    def forward(self, input_ids):
        emb = self.word_embeddings(input_ids)
        return self.classifier(emb.mean(dim=1))

model = ToyModel()

# Simulate a forward + backward pass
input_ids = torch.randint(0, 100, (2, 5))
output = model(input_ids)
loss = output.sum()
loss.backward()

# --- FGM attack ---
fgm = FGM(model, epsilon=1.0)
original_emb = model.word_embeddings.weight.data.clone()

fgm.attack()
perturbed_emb = model.word_embeddings.weight.data.clone()
perturbation_norm = torch.norm(perturbed_emb - original_emb).item()
print(f"FGM perturbation L2 norm: {perturbation_norm:.4f}")

fgm.restore()
restored_emb = model.word_embeddings.weight.data.clone()
print(f"Embeddings restored: {torch.allclose(original_emb, restored_emb)}")

In [ ]:
# --- PGD multi-step attack ---
model.zero_grad()
output = model(input_ids)
loss = output.sum()
loss.backward()

pgd = PGD(model, epsilon=0.3, alpha=0.1, num_steps=3)
original_emb = model.word_embeddings.weight.data.clone()

pgd.save()
for step in range(pgd.num_steps):
    pgd.attack_step()
    step_emb = model.word_embeddings.weight.data.clone()
    step_norm = torch.norm(step_emb - original_emb).item()
    print(f"  PGD step {step+1}: perturbation L2 norm = {step_norm:.4f}")

pgd.restore()
print(f"Embeddings restored: {torch.allclose(original_emb, model.word_embeddings.weight.data)}")

In [ ]:
# --- AdversarialTrainer overview ---
print("AdversarialTrainer extends HuggingFace Trainer with:")
print("  - Automatic FGM or PGD perturbation during training_step()")
print("  - Clean loss + adversarial loss combined")
print("  - No architecture changes required")
print()
print("Usage:")
print("  trainer = AdversarialTrainer(")
print("      model=model, args=training_args,")
print("      train_dataset=train_ds, eval_dataset=eval_ds,")
print("      adv_method='fgm', adv_epsilon=1.0,")
print("  )")
print("  trainer.train()")
print()
print("CLI:")
print("  python scripts/train.py --model pubmedbert --dataset icd_ner --adversarial")
print("  python scripts/train.py --model pubmedbert --dataset icd_ner --adversarial --adv-method pgd")

---
## 12. Assertion Classifier (Transformer-based) — Overview

The `AssertionClassifier` uses `bvanaken/clinical-assertion-negation-bert` for learned assertion detection.
It downloads the model on first use (~440MB), so we show the API without executing.

In [ ]:
# To actually run: uncomment the lines below (requires model download)
#
# from src.clinical.assertion import AssertionClassifier
#
# classifier = AssertionClassifier(device="cpu")
#
# result = classifier.predict(
#     text="Patient denies any chest pain or shortness of breath.",
#     entity_text="chest pain",
#     entity_start=19,
#     entity_end=29,
# )
# print(result)  # {'label': 'ABSENT', 'negation': 'negated', 'score': 0.97}
#
# # Batch annotation:
# entities = [
#     {"text": "chest pain", "label": "DIAGNOSIS", "start": 19, "end": 29},
# ]
# annotated = classifier.annotate_entities(
#     "Patient denies any chest pain.", entities
# )

print("AssertionClassifier API:")
print("  .predict(text, entity_text, entity_start, entity_end) → Dict")
print("    Returns: {'label': 'PRESENT'|'ABSENT'|'POSSIBLE', 'negation': ..., 'score': float}")
print()
print("  .annotate_entities(text, entities) → List[Dict]")
print("    Adds: 'negation', 'assertion_label', 'assertion_score' to each entity")
print()
print("Pipeline integration:")
print("  pipeline = MedicalCodingPipeline(negation_strategy='transformer')")

---
## 13. Dataset Loading & Preprocessing

Load a biomedical NER dataset, inspect its structure, build label maps, and tokenize for training.

In [ ]:
from src.data.dataset_loader import load_ner_dataset, get_label_maps
from src.data.preprocessing import preprocess_dataset, tokenize_and_align_labels

# Load the NCBI Disease corpus (smallest, fastest to download)
dataset, label_list = load_ner_dataset("ncbi_disease")

print(f"=== NCBI Disease Corpus ===")
print(f"Label scheme: {label_list}")
print()
for split in dataset:
    print(f"  {split:12s}: {len(dataset[split]):,} examples")

# Inspect a raw example
ex = dataset["train"][0]
print(f"\nSample example:")
print(f"  Tokens:     {ex['tokens'][:12]}...")
print(f"  NER labels: {ex['ner_labels'][:12]}...")
print(f"  Label IDs:  {ex['ner_tags'][:12]}...")

label2id, id2label = get_label_maps(label_list)
print(f"\nLabel mapping: {label2id}")

In [ ]:
from transformers import AutoTokenizer

# Load a small, fast tokenizer for demonstration
tokenizer = AutoTokenizer.from_pretrained(
    "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext",
    use_fast=True,
)

# Tokenize a single example to show subword alignment
sample = {"tokens": [ex["tokens"]], "ner_labels": [ex["ner_labels"]]}
aligned = tokenize_and_align_labels(sample, tokenizer, label2id, max_length=128)

input_ids = aligned["input_ids"][0]
labels = aligned["labels"][0]

# Show the alignment: subword tokens ↔ aligned labels
subwords = tokenizer.convert_ids_to_tokens(input_ids)
print("=== Subword ↔ Label Alignment (first 25 tokens) ===")
print(f"{'Subword':<20s} {'Label ID':>8s}  {'Label':>15s}")
print("-" * 48)
for sw, lab in zip(subwords[:25], labels[:25]):
    lab_str = id2label[lab] if lab != -100 else "(ignored)"
    print(f"  {sw:<20s} {lab:>6d}  {lab_str:>15s}")
print(f"\n  ... ({len(subwords)} subwords total, original: {len(ex['tokens'])} words)")

In [ ]:
# Tokenize the full dataset for training
tokenized = preprocess_dataset(
    dataset, tokenizer, label2id,
    max_length=128,   # shorter for demo speed
    num_proc=1,       # single process for notebook stability
)

for split in tokenized:
    print(f"  {split:12s}: {len(tokenized[split]):,} examples, "
          f"columns={list(tokenized[split].column_names)}")

---
## 14. Model Building & Training

Build a PubMedBERT token classifier, configure training with the standard hyperparameters, and train on a small subset to demonstrate the full training loop.

> **Note:** This trains for 2 epochs on 200 examples as a demonstration. A production run uses 20 epochs on the full dataset (~7K+ examples for `icd_ner`).

In [ ]:
from configs.ner_config import NERConfig
from src.models.ner_model import build_ner_model
from src.training.trainer import build_trainer

# Build a compact training config for demonstration
config = NERConfig(
    model_key="pubmedbert",
    dataset_key="ncbi_disease",
    num_train_epochs=2,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    learning_rate=5e-5,
    warmup_ratio=0.1,
    max_seq_length=128,
    fp16=False,               # CPU-safe
    logging_steps=10,
    eval_steps=50,
    save_steps=50,
    save_total_limit=1,
    early_stopping_patience=3,
    output_dir="outputs",
    use_adversarial_training=False,
)

print(f"Experiment: {config.experiment_name}")
print(f"Model:      {config.model_name_or_path}")
print(f"LR={config.learning_rate}, Epochs={config.num_train_epochs}, "
      f"Batch={config.per_device_train_batch_size}")

In [ ]:
# Build model + tokenizer
model, tokenizer = build_ner_model(
    config.model_name_or_path,
    label_list,
    use_crf=config.use_crf,
)

# Count parameters
total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Parameters: {trainable:,} trainable / {total:,} total")
print(f"Labels:     {model.config.num_labels} ({label_list})")

In [ ]:
# Tokenize with the model's own tokenizer
tokenized = preprocess_dataset(
    dataset, tokenizer, label2id,
    max_length=config.max_seq_length,
    num_proc=1,
)

# Use a small subset for fast demo training
train_subset = tokenized["train"].select(range(min(200, len(tokenized["train"]))))
eval_split = "validation" if "validation" in tokenized else "test"
eval_subset = tokenized[eval_split].select(range(min(100, len(tokenized[eval_split]))))

print(f"Training on {len(train_subset)} examples, evaluating on {len(eval_subset)}")

In [ ]:
# Build the HuggingFace Trainer
trainer = build_trainer(
    model=model,
    tokenizer=tokenizer,
    config=config,
    train_dataset=train_subset,
    eval_dataset=eval_subset,
    label_list=label_list,
)

print(f"Trainer type: {type(trainer).__name__}")
print(f"Callbacks: {[type(cb).__name__ for cb in trainer.callback_handler.callbacks]}")

In [ ]:
# Train!
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

print("=== Training ===")
train_result = trainer.train()

print(f"\n=== Training Complete ===")
print(f"  Total steps:    {train_result.global_step}")
print(f"  Training loss:  {train_result.training_loss:.4f}")
for key, val in train_result.metrics.items():
    print(f"  {key}: {val}")

In [ ]:
# Save the trained model
import os

best_model_dir = os.path.join(config.output_dir, config.experiment_name, "best_model")
trainer.save_model(best_model_dir)
tokenizer.save_pretrained(best_model_dir)
print(f"Model saved to: {best_model_dir}")
print(f"Contents: {os.listdir(best_model_dir)}")

---
## 15. Evaluation & Error Analysis

Evaluate the trained model on the held-out set and run entity-level error analysis to identify boundary errors, false negatives, and false positives.

In [ ]:
# Evaluate on the held-out split
eval_metrics = trainer.evaluate()

print("=== Evaluation Results ===")
for key, val in sorted(eval_metrics.items()):
    if isinstance(val, float):
        print(f"  {key:30s}: {val:.4f}")
    else:
        print(f"  {key:30s}: {val}")

In [ ]:
# Run NER predictions on raw examples for error analysis
from src.inference.predictor import NERPredictor

predictor = NERPredictor(model_path=best_model_dir, device="cpu")

# Predict on a handful of evaluation examples
n_eval = min(50, len(dataset[eval_split]))
eval_examples = [dataset[eval_split][i] for i in range(n_eval)]

tokens_list = []
true_labels_list = []
pred_labels_list = []

for ex in eval_examples:
    tokens = ex["tokens"]
    true_labels = ex["ner_labels"]
    pred_labels, _ = predictor.predict_tokens(tokens)
    tokens_list.append(tokens)
    true_labels_list.append(true_labels)
    pred_labels_list.append(pred_labels)

print(f"Predicted {n_eval} examples for error analysis")

# Show a few predictions side-by-side
print("\n=== Sample Predictions ===")
for i in range(min(3, n_eval)):
    tokens = tokens_list[i]
    true = true_labels_list[i]
    pred = pred_labels_list[i]
    # Show only tokens with non-O labels
    interesting = [(t, tr, pr) for t, tr, pr in zip(tokens, true, pred)
                   if tr != "O" or pr != "O"]
    if interesting:
        print(f"\n  Example {i+1}:")
        for tok, tr, pr in interesting[:10]:
            match = "  " if tr == pr else "!!"
            print(f"    {match} {tok:25s} true={tr:20s} pred={pr}")

In [ ]:
from src.evaluation.error_analysis import analyse_errors, print_error_report

analysis = analyse_errors(tokens_list, true_labels_list, pred_labels_list)

report = print_error_report(analysis, top_k=10)
print(report)

---
## 16. Full Pipeline — Shorthand to Cost Estimate

Run the complete six-stage pipeline on realistic clinical notes using the model we just trained. This demonstrates the full power of the system: abbreviation expansion, NER extraction, negation detection, ICD-10-CM coding, and MS-DRG cost estimation.

In [ ]:
from src.clinical.pipeline import MedicalCodingPipeline

# Initialize the full pipeline with our trained model
full_pipeline = MedicalCodingPipeline(
    model_path=best_model_dir,
    expand_shorthand=True,
    detect_negation=True,
    negation_strategy="rules",
    resolve_icd_codes=True,
    icd_top_k=3,
    resolve_drg=True,         # Enable DRG cost estimation (NBER CMS Table 5 weights)
    device="cpu",
)

print("=== Full Pipeline Components ===")
print(f"  NER model:          {best_model_dir}")
print(f"  Shorthand expander: {full_pipeline.shorthand_expander is not None}")
print(f"  Negation detector:  {full_pipeline.negation_detector is not None}")
print(f"  ICD lookup:         {full_pipeline.icd_lookup is not None} "
      f"({len(full_pipeline.icd_lookup._codes):,} codes)" if full_pipeline.icd_lookup else "")
print(f"  DRG estimator:      {full_pipeline.drg_estimator is not None} "
      f"({full_pipeline.drg_estimator.num_drgs} DRGs, CMS FY 2026 weights)"
      if full_pipeline.drg_estimator else "")

In [ ]:
# --- Clinical Note 1: Shorthand-heavy emergency note ---
note1 = "Pt c/o sob and cp x 2 days. Hx of chf and dm2. Denies n/v. Afebrile, bp stable."

print("=" * 70)
print("CLINICAL NOTE 1 (Emergency)")
print("=" * 70)
print(f"Input:  {note1}")

# Show shorthand expansion step
expanded, offsets = full_pipeline.shorthand_expander.expand_with_offsets(note1)
print(f"\nExpanded: {expanded}")
if offsets:
    print(f"  Expansions: {len(offsets)}")
    for om in offsets[:5]:
        print(f"    '{om['abbreviation']}' -> '{om['expansion']}'")

# Run full pipeline
results1 = full_pipeline.process(note1)
print(f"\n--- Extracted Entities ({len(results1)}) ---")
for ent in results1:
    icd_str = ent.icd_codes[0]['code'] + ": " + ent.icd_codes[0]['description'][:35] if ent.icd_codes else "N/A"
    abbrev = f" (from '{ent.expanded_from}')" if ent.expanded_from else ""
    print(f"  [{ent.text}]{abbrev}")
    print(f"    Assertion: {ent.negation.upper():12s} | Score: {ent.score:.3f} | ICD: {icd_str}")

In [ ]:
# --- Clinical Note 2: Complex admission with negations ---
note2 = (
    "72 yo male admitted with acute exacerbation of COPD and pneumonia. "
    "History of atrial fibrillation and chronic kidney disease stage 3. "
    "No evidence of pulmonary embolism on CT angiography. "
    "Patient denies chest pain or palpitations."
)

print("=" * 70)
print("CLINICAL NOTE 2 (Admission)")
print("=" * 70)
print(f"Input: {note2}\n")

results2 = full_pipeline.process(note2)
print(f"--- Extracted Entities ({len(results2)}) ---")
for ent in results2:
    parts = [f"{ent.negation.upper():12s}"]
    if ent.negation_trigger:
        parts.append(f'trigger="{ent.negation_trigger}"')
    if ent.icd_codes:
        parts.append(f"ICD={ent.icd_codes[0]['code']}")
    if ent.drg_info:
        parts.append(f"DRG={ent.drg_info.get('current', {}).get('drg_code', 'N/A')}")
    annotation = " | ".join(parts)
    print(f"  [{ent.text:30s}] {annotation}")

# Show DRG cost analysis if available
drg_entities = [e for e in results2 if e.drg_info]
if drg_entities:
    drg = drg_entities[0].drg_info
    print(f"\n--- DRG Cost Analysis ---")
    current = drg.get("current", {})
    print(f"  DRG {current.get('drg_code', 'N/A')}: {current.get('drg_title', 'N/A')}")
    print(f"  Estimated payment: ${current.get('estimated_payment', 0):,.2f}")
    print(f"  Revenue at risk:   ${drg.get('revenue_at_risk', 0):,.2f}")
    print(f"  Undercoding risk:  {drg.get('undercoding_risk', False)}")

In [ ]:
# --- Clinical Note 3: Family history and hypotheticals ---
note3 = (
    "Patient presents with severe headache and fever. "
    "Family history of breast cancer and stroke. "
    "If symptoms persist, consider meningitis workup. "
    "Possible migraine vs tension headache."
)

print("=" * 70)
print("CLINICAL NOTE 3 (Outpatient)")
print("=" * 70)
print(f"Input: {note3}\n")

results3 = full_pipeline.process(note3)
print(f"--- Assertion Classification ({len(results3)} entities) ---")
for ent in results3:
    icd = ent.icd_codes[0]['code'] if ent.icd_codes else "N/A"
    print(f"  {ent.negation.upper():14s} [{ent.text:25s}] ICD={icd}")

In [ ]:
# --- Batch processing ---
clinical_notes = [
    "Pt denies cp. Dx with afib and htn.",
    "No fever. History of diabetes. Acute kidney injury on admission.",
    "Possible sepsis. Blood cultures pending. Started on broad spectrum abx.",
]

print("=" * 70)
print("BATCH PROCESSING (3 notes)")
print("=" * 70)

batch_results = full_pipeline.process_batch(clinical_notes)
for i, (note, entities) in enumerate(zip(clinical_notes, batch_results)):
    print(f"\nNote {i+1}: {note}")
    if entities:
        for ent in entities:
            icd = ent.icd_codes[0]['code'] if ent.icd_codes else "N/A"
            print(f"  -> [{ent.text}] {ent.negation.upper()} ICD={icd}")
    else:
        print("  -> (no entities extracted)")

In [ ]:
# --- JSON export for downstream systems ---
import json

class NumpyEncoder(json.JSONEncoder):
    """Handle numpy float32 from model scores."""
    def default(self, obj):
        import numpy as np
        if isinstance(obj, (np.floating, np.integer)):
            return float(obj)
        return super().default(obj)

print("=== JSON Export (Note 2) ===")
export = {
    "text": note2,
    "entities": [ent.to_dict() for ent in results2],
    "summary": {
        "total_entities": len(results2),
        "affirmed": sum(1 for e in results2 if e.is_affirmed),
        "negated": sum(1 for e in results2 if e.is_negated),
        "historical": sum(1 for e in results2 if e.negation == "historical"),
        "icd_codes_resolved": sum(1 for e in results2 if e.icd_codes),
    }
}
print(json.dumps(export, indent=2, cls=NumpyEncoder)[:2000])

---
## 17. Multi-Model Training on the Full ICD Dataset

Train multiple biomedical language models on the **full** 7-source ICD composite dataset (NCBI Disease + BC5CDR + BioMed NER + ADE Corpus + Curated + MedMentions + MACCROBAT). Training uses the **combined train + validation splits** for maximum data, with the held-out **test split** used for final evaluation.

**Training setup:**
- **Data**: Full train + validation merged as training data; test split held out
- **Epochs**: 10 (with early stopping patience=5)
- **No adversarial training** — clean baselines for comparison
- **Evaluation**: Entity-level Precision / Recall / F1 on the test split

Models compared:
- **PubMedBERT** — SOTA on BLURB benchmark (PubMed abstracts + full text)
- **BioBERT** — Pre-trained on PubMed abstracts
- **Bio_ClinicalBERT** — Pre-trained on MIMIC-III clinical notes

In [ ]:
# --- Load the full ICD composite dataset (all 7 sources) ---
from src.data.icd_dataset import load_icd_ner_dataset
from src.data.preprocessing import preprocess_dataset
from src.data.dataset_loader import get_label_maps
from datasets import concatenate_datasets, DatasetDict

icd_dataset, icd_label_list = load_icd_ner_dataset()

print("=== ICD NER Composite Dataset (7 Sources) ===")
print(f"Label scheme: {icd_label_list}")
print()
for split in icd_dataset:
    n_ents = sum(1 for ex in icd_dataset[split] for lab in ex["ner_labels"] if lab.startswith("B-"))
    print(f"  {split:12s}: {len(icd_dataset[split]):>6,} examples, {n_ents:>6,} DIAGNOSIS entities")

# Merge train + validation for training; hold out test for final eval
train_data = icd_dataset["train"]
if "validation" in icd_dataset:
    val_data = icd_dataset["validation"]
    merged_train = concatenate_datasets([train_data, val_data])
    print(f"\n  Merged train+val: {len(merged_train):>6,} examples (for training)")
else:
    merged_train = train_data
    print(f"\n  Training on train split: {len(merged_train):>6,} examples")

test_data = icd_dataset["test"]
print(f"  Held-out test:    {len(test_data):>6,} examples (for evaluation)")

# Build a DatasetDict with merged train and test
training_dataset = DatasetDict({
    "train": merged_train,
    "test": test_data,
})

icd_label2id, icd_id2label = get_label_maps(icd_label_list)
print(f"\nTotal: {sum(len(icd_dataset[s]) for s in icd_dataset):,} examples across all original splits")

In [ ]:
# --- Train three biomedical models on the FULL ICD dataset ---
import warnings, time, torch
from configs.ner_config import NERConfig, MODEL_CONFIGS
from src.models.ner_model import build_ner_model
from src.training.trainer import build_trainer
from src.data.preprocessing import preprocess_dataset
from src.data.dataset_loader import get_label_maps

warnings.filterwarnings("ignore", category=FutureWarning)

# Models to compare (all 110M params, different pre-training domains)
model_keys = ["pubmedbert", "biobert", "bio_clinicalbert"]

# Detect device
device_name = "cuda" if torch.cuda.is_available() else "cpu"
use_fp16 = torch.cuda.is_available()
print(f"Device: {device_name} | FP16: {use_fp16}")

# Store results for comparison
icd_results = {}

for model_key in model_keys:
    print("=" * 70)
    print(f"  Training: {model_key} ({MODEL_CONFIGS[model_key]['description'][:50]})")
    print("=" * 70)

    # Configure for ICD dataset — full training pipeline
    cfg = NERConfig(
        model_key=model_key,
        dataset_key="icd_ner",
        num_train_epochs=10,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        learning_rate=5e-5,
        warmup_ratio=0.1,
        weight_decay=0.01,
        max_seq_length=512,
        fp16=use_fp16,
        logging_steps=50,
        eval_steps=200,
        save_steps=200,
        save_total_limit=1,
        early_stopping_patience=5,
        output_dir="outputs",
        use_adversarial_training=False,
    )

    # Build model + tokenizer
    mdl, tok = build_ner_model(cfg.model_name_or_path, icd_label_list, use_crf=False)
    n_params = sum(p.numel() for p in mdl.parameters() if p.requires_grad)
    print(f"  Parameters: {n_params:,}")

    # Tokenize the full training dataset (train+val merged) and test set
    icd_l2id, icd_i2l = get_label_maps(icd_label_list)
    tok_ds = preprocess_dataset(training_dataset, tok, icd_l2id, max_length=cfg.max_seq_length, num_proc=1)

    # Build the HuggingFace Trainer
    save_dir = f"outputs/{model_key}_icd_ner_full/best_model"
    trainer = build_trainer(
        model=mdl,
        tokenizer=tok,
        train_dataset=tok_ds["train"],
        eval_dataset=tok_ds["test"],  # Evaluate on held-out test
        label_list=icd_label_list,
        training_args_kwargs={
            "output_dir": f"outputs/{model_key}_icd_ner_full",
            "num_train_epochs": cfg.num_train_epochs,
            "per_device_train_batch_size": cfg.per_device_train_batch_size,
            "per_device_eval_batch_size": cfg.per_device_eval_batch_size,
            "learning_rate": cfg.learning_rate,
            "warmup_ratio": cfg.warmup_ratio,
            "weight_decay": cfg.weight_decay,
            "fp16": cfg.fp16,
            "logging_steps": cfg.logging_steps,
            "eval_strategy": "steps",
            "eval_steps": cfg.eval_steps,
            "save_strategy": "steps",
            "save_steps": cfg.save_steps,
            "save_total_limit": cfg.save_total_limit,
            "load_best_model_at_end": True,
            "metric_for_best_model": "f1",
            "greater_is_better": True,
        },
    )

    # Train on the full train+val merged set
    t0 = time.time()
    train_result = trainer.train()
    elapsed = time.time() - t0

    # Evaluate on held-out test set
    eval_metrics = trainer.evaluate()

    # Save model
    trainer.save_model(save_dir)
    tok.save_pretrained(save_dir)

    icd_results[model_key] = {
        "f1": eval_metrics.get("eval_f1", 0),
        "precision": eval_metrics.get("eval_precision", 0),
        "recall": eval_metrics.get("eval_recall", 0),
        "eval_loss": eval_metrics.get("eval_loss", 0),
        "time_s": elapsed,
        "save_dir": save_dir,
        "n_params": n_params,
        "train_samples": len(tok_ds["train"]),
        "test_samples": len(tok_ds["test"]),
    }

    print(f"  Test F1: {eval_metrics.get('eval_f1', 0):.4f} | "
          f"P: {eval_metrics.get('eval_precision', 0):.4f} | "
          f"R: {eval_metrics.get('eval_recall', 0):.4f} | "
          f"Time: {elapsed:.1f}s")
    print(f"  Saved to: {save_dir}\n")

In [ ]:
# --- Compare model results on the held-out TEST set ---
print("=" * 78)
print("  MULTI-MODEL COMPARISON — ICD NER (Tested on Held-Out Test Set)")
print("=" * 78)
print()
print(f"{'Model':<20s} {'Test F1':>8s} {'Precision':>10s} {'Recall':>8s} {'Loss':>8s} "
      f"{'Train':>7s} {'Test':>6s} {'Time':>8s}")
print("-" * 78)

best_f1 = 0
best_model = ""
for model_key in model_keys:
    r = icd_results[model_key]
    if r["f1"] > best_f1:
        best_f1 = r["f1"]
        best_model = model_key
    print(f"  {model_key:<18s} {r['f1']:>7.4f} {r['precision']:>9.4f} {r['recall']:>7.4f} "
          f"{r['eval_loss']:>7.4f} {r['train_samples']:>6,} {r['test_samples']:>5,} "
          f"{r['time_s']:>6.1f}s")

print(f"\nBest model: {best_model} (Test F1={best_f1:.4f})")
print(f"  Saved to: {icd_results[best_model]['save_dir']}")
print(f"\nTraining: 10 epochs, lr=5e-5, batch=16, max_len=512, early_stop=5")
print(f"Data: train+validation merged → training | test → evaluation")

---
## 18. ICD-Trained Pipeline — Shorthand, Negation & DRG Cost Analysis

Use the best model from the multi-model comparison to run the complete six-stage pipeline on clinical text, including **real examples from the test dataset** and abbreviation-heavy clinical notes:

1. **Shorthand expansion** — physician abbreviations decoded with offset tracking
2. **NER extraction** — DIAGNOSIS entities extracted by the ICD-trained model
3. **Negation / assertion** — rule-based ConText/NegEx (affirmed, negated, historical, family, possible, hypothetical)
4. **ICD-10-CM coding** — TF-IDF entity linking against 51K codes
5. **MS-DRG grouping** — drgpy ICD code set → DRG (all 767 MS-DRGs)
6. **Cost impact analysis** — estimated Medicare payment and revenue-at-risk quantification

In [ ]:
# --- Build full pipeline with the best ICD-trained model ---
from src.clinical.pipeline import MedicalCodingPipeline

# Use the best model from the Section 17 comparison
best_icd_model_dir = icd_results[best_model]["save_dir"]

icd_pipeline = MedicalCodingPipeline(
    model_path=best_icd_model_dir,
    expand_shorthand=True,       # Stage 1: abbreviation expansion
    detect_negation=True,        # Stage 3: negation / assertion detection
    negation_strategy="rules",   # ConText/NegEx (100+ triggers, 6 statuses)
    resolve_icd_codes=True,      # Stage 4: ICD-10-CM entity linking
    icd_top_k=3,
    resolve_drg=True,            # Stage 5-6: DRG grouping + cost analysis (NBER CMS weights)
    device="cpu",
)

print(f"=== ICD-Trained Full Pipeline ===")
print(f"  Best model:         {best_model} (Test F1={best_f1:.4f})")
print(f"  Model path:         {best_icd_model_dir}")
print(f"  Shorthand expander: {icd_pipeline.shorthand_expander is not None}")
print(f"  Negation detector:  {icd_pipeline.negation_detector is not None}")
print(f"  ICD lookup:         {icd_pipeline.icd_lookup is not None} "
      f"({len(icd_pipeline.icd_lookup._codes):,} codes)")
print(f"  DRG estimator:      {icd_pipeline.drg_estimator is not None} "
      f"({icd_pipeline.drg_estimator.num_drgs} DRGs, CMS FY 2026 weights from NBER)")

In [ ]:
# --- Case 1: Test dataset examples through the pipeline ---
# Pull real sentences from the held-out test set and run the full pipeline.

print("=" * 78)
print("  CASE 1: Real Test Dataset Examples Through Full Pipeline")
print("=" * 78)

# Select test examples that contain DIAGNOSIS entities
test_examples = []
for i, ex in enumerate(icd_dataset["test"]):
    entities = [t for t, l in zip(ex["tokens"], ex["ner_labels"]) if l.startswith("B-")]
    if len(entities) >= 2:
        test_examples.append((i, ex))
    if len(test_examples) >= 5:
        break

for idx, (i, ex) in enumerate(test_examples):
    text = " ".join(ex["tokens"])
    gold_entities = [t for t, l in zip(ex["tokens"], ex["ner_labels"]) if l.startswith("B-")]

    print(f"\n  Test example [{i}]: {text[:100]}...")
    print(f"    Gold entities: {gold_entities}")

    # Run through full pipeline
    results = icd_pipeline.process(text)

    print(f"    Predicted ({len(results)} entities):")
    for ent in results:
        icd_code = ent.icd_codes[0]["code"] if ent.icd_codes else "—"
        print(f"      {ent.text:<30s} {ent.negation.upper():<12s} ICD={icd_code}")

    # DRG info if available
    drg_ents = [e for e in results if e.drg_info]
    if drg_ents:
        drg = drg_ents[0].drg_info
        curr = drg.get("current", {})
        print(f"    DRG: {curr.get('drg_code', 'N/A')} — {curr.get('drg_title', 'N/A')[:50]}")
        print(f"    Payment: ${curr.get('estimated_payment', 0):,.2f}")

In [ ]:
# --- Case 2: Shorthand-heavy clinical note with full DRG analysis ---
note_clinical = (
    "72 yo M pt c/o sob and cp x 3 days. Hx of chf, dm2, and ckd. "
    "Denies n/v or fever. Dx: acute exacerbation copd w/ pna. "
    "r/o pe. Htn controlled on meds."
)

print("=" * 78)
print("  CASE 2: Shorthand-Heavy Clinical Note — Full Pipeline")
print("=" * 78)
print(f"\n  Raw input:\n    {note_clinical}\n")

# Stage 1: Shorthand expansion
expanded, offsets = icd_pipeline.shorthand_expander.expand_with_offsets(note_clinical)
print("  STAGE 1 — Shorthand Expansion:")
print(f"    Expanded: {expanded}")
print(f"    Abbreviations decoded ({len(offsets)}):")
for om in offsets:
    print(f"      '{om['abbreviation']:6s}' → '{om['expansion']}'")

# Full pipeline
results_clin = icd_pipeline.process(note_clinical)

# Stages 2-4
print(f"\n  STAGE 2-4 — NER + Assertion + ICD ({len(results_clin)} entities):")
print(f"    {'Entity':<30s} {'Assertion':<14s} {'ICD Code':<10s} {'ICD Description':<35s}")
print("    " + "-" * 90)
for ent in results_clin:
    icd_code = ent.icd_codes[0]["code"] if ent.icd_codes else "—"
    icd_desc = ent.icd_codes[0]["description"][:33] if ent.icd_codes else "—"
    abbrev = f" (← {ent.expanded_from})" if ent.expanded_from else ""
    print(f"    {(ent.text + abbrev):<30s} {ent.negation.upper():<14s} {icd_code:<10s} {icd_desc}")

# Stage 5-6: DRG
affirmed_codes = [ent.icd_codes[0]["code"] for ent in results_clin
                  if ent.is_affirmed and ent.icd_codes]
print(f"\n  STAGE 5-6 — DRG Cost Analysis:")
print(f"    Affirmed codes: {affirmed_codes}")

drg_ents = [e for e in results_clin if e.drg_info]
if drg_ents:
    drg = drg_ents[0].drg_info
    curr = drg.get("current", {})
    print(f"    Assigned DRG:      {curr.get('drg_code', 'N/A')} — {curr.get('drg_title', 'N/A')}")
    print(f"    Relative weight:   {curr.get('relative_weight', 'N/A')}")
    print(f"    Estimated payment: ${curr.get('estimated_payment', 0):,.2f}")
    print(f"    Severity level:    {curr.get('severity_level', 'N/A')}")
    if drg.get("undercoding_risk"):
        print(f"    Revenue at risk:   ${drg.get('revenue_at_risk', 0):,.2f}")
else:
    print("    (No DRG assignment available)")

In [ ]:
# --- Case 3: Sepsis admission — high-acuity DRG with comorbidities ---
note_sepsis = (
    "58 yo M presents to ED w/ fever, tachycardia, and hypotension. "
    "Dx: sepsis secondary to uti. Pmhx of dm2, chf, and ckd stage 4. "
    "No hx of mi or stroke. Denies cp. Started on abx."
)

print("=" * 78)
print("  CASE 3: Sepsis Admission — High-Acuity DRG")
print("=" * 78)
print(f"\n  Raw input:\n    {note_sepsis}\n")

# Full pipeline
results_sepsis = icd_pipeline.process(note_sepsis)

# Combined view
print(f"  STAGE 2-4 — Entity Extraction, Assertion & ICD Coding:")
print(f"    {'Entity':<28s} {'Assertion':<14s} {'ICD Code':<10s} {'ICD Description':<35s}")
print("    " + "-" * 90)
for ent in results_sepsis:
    icd_code = ent.icd_codes[0]["code"] if ent.icd_codes else "—"
    icd_desc = ent.icd_codes[0]["description"][:33] if ent.icd_codes else "—"
    print(f"    {ent.text:<28s} {ent.negation.upper():<14s} {icd_code:<10s} {icd_desc}")

# DRG cost
affirmed_sepsis = [e for e in results_sepsis if e.is_affirmed]
affirmed_codes_sepsis = [e.icd_codes[0]["code"] for e in affirmed_sepsis if e.icd_codes]
print(f"\n  STAGE 5-6 — DRG Cost Analysis:")
print(f"    Affirmed codes: {affirmed_codes_sepsis}")

drg_ents = [e for e in results_sepsis if e.drg_info]
if drg_ents:
    drg = drg_ents[0].drg_info
    curr = drg.get("current", {})
    print(f"    Assigned DRG:      {curr.get('drg_code', 'N/A')} — {curr.get('drg_title', 'N/A')}")
    print(f"    Relative weight:   {curr.get('relative_weight', 'N/A')}")
    print(f"    Estimated payment: ${curr.get('estimated_payment', 0):,.2f}")
    print(f"    Severity level:    {curr.get('severity_level', 'N/A')}")
    if drg.get("undercoding_risk"):
        print(f"    Revenue at risk:   ${drg.get('revenue_at_risk', 0):,.2f}")
else:
    print("    (No DRG assignment available)")

In [ ]:
# --- Cross-case summary: pipeline outcomes at a glance ---
import json, numpy as np

class NumpyEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, (np.floating, np.integer)):
            return float(obj)
        return super().default(obj)

cases = [
    ("Case 2: Clinical / COPD+Pneumonia", note_clinical, results_clin),
    ("Case 3: Sepsis / High-Acuity", note_sepsis, results_sepsis),
]

print("=" * 78)
print("  CROSS-CASE SUMMARY — Full Pipeline Outcomes")
print("=" * 78)
print()
print(f"  {'Case':<35s} {'Entities':>8s} {'Affirm':>7s} {'Neg':>5s} "
      f"{'Other':>6s} {'DRG':>6s} {'Payment':>12s}")
print("  " + "-" * 82)

for case_name, note, results in cases:
    n_total = len(results)
    n_affirm = sum(1 for e in results if e.is_affirmed)
    n_neg = sum(1 for e in results if e.is_negated)
    n_other = n_total - n_affirm - n_neg
    drg_ents = [e for e in results if e.drg_info]
    if drg_ents:
        drg_code = drg_ents[0].drg_info.get("current", {}).get("drg_code", "—")
        payment = drg_ents[0].drg_info.get("current", {}).get("estimated_payment", 0)
        payment_str = f"${payment:>10,.2f}"
    else:
        drg_code = "—"
        payment_str = "—"
    print(f"  {case_name:<35s} {n_total:>8d} {n_affirm:>7d} {n_neg:>5d} "
          f"{n_other:>6d} {drg_code:>6s} {payment_str:>12s}")

# JSON export of a case
print(f"\n{'=' * 78}")
print("  JSON Export — Case 2 (first entity)")
print(f"{'=' * 78}")
if results_clin:
    print(json.dumps(results_clin[0].to_dict(), indent=2, cls=NumpyEncoder))

---
## 19. GatorTron QLoRA — Parameter-Efficient Fine-Tuning

Fine-tune the **GatorTron-base** model (345M params, trained on 90B words of clinical text) using **QLoRA** (4-bit quantization + LoRA adapters). This trains only ~0.5% of parameters while matching full fine-tuning performance.

**Why QLoRA for GatorTron?**
- GatorTron at 345M params needs ~2.5x more memory than 110M models
- QLoRA reduces memory by ~87.5% via 4-bit NF4 quantization
- LoRA adapters on attention layers train only ~1.8M parameters
- The NER classification head trains in full precision (`modules_to_save=["classifier"]`)

**Requirements:** `peft>=0.6.0`, `bitsandbytes>=0.41.0` (CUDA GPU)

> **Note:** LoRA (without quantization) works on CPU and is the recommended default. QLoRA requires a CUDA GPU. For 110M models (PubMedBERT, BioBERT), standard full fine-tuning is sufficient.

In [ ]:
# --- GatorTron QLoRA / LoRA configuration ---
# This cell demonstrates LoRA setup. To use QLoRA (4-bit), set use_qlora=True
# and ensure bitsandbytes is installed with a CUDA GPU.

from src.models.ner_model import build_ner_model
from configs.ner_config import NERConfig, MODEL_CONFIGS
import torch

gatortron_config = NERConfig(
    model_key="gatortron-base",
    dataset_key="icd_ner",
    use_lora=True,             # Enable LoRA adapters
    use_qlora=False,           # Set True for 4-bit quantization (needs CUDA + bitsandbytes)
    lora_r=16,                 # LoRA rank
    lora_alpha=16,             # Scaling factor (alpha/r = 1.0)
    lora_dropout=0.1,
    lora_target_modules="query,key,value",  # MegatronBERT attention layers
    num_train_epochs=10,
    per_device_train_batch_size=16,
    learning_rate=1e-3,        # Higher LR for LoRA (vs 5e-5 for full fine-tuning)
    warmup_ratio=0.1,
    fp16=torch.cuda.is_available(),
)

print(f"=== GatorTron LoRA Configuration ===")
print(f"  Model:          {gatortron_config.model_name_or_path}")
print(f"  LoRA:           r={gatortron_config.lora_r}, alpha={gatortron_config.lora_alpha}")
print(f"  QLoRA (4-bit):  {gatortron_config.use_qlora}")
print(f"  Target modules: {gatortron_config.lora_target_modules}")
print(f"  Learning rate:  {gatortron_config.learning_rate} (higher for LoRA)")
print(f"  Device:         {'CUDA' if torch.cuda.is_available() else 'CPU'}")

In [ ]:
# --- Build GatorTron with LoRA adapters ---
# This loads the full 345M model and applies LoRA, resulting in ~1.8M trainable params.

lora_targets = gatortron_config.lora_target_modules.split(",")

gatortron_model, gatortron_tok = build_ner_model(
    gatortron_config.model_name_or_path,
    icd_label_list,
    use_lora=gatortron_config.use_lora,
    use_qlora=gatortron_config.use_qlora,
    lora_r=gatortron_config.lora_r,
    lora_alpha=gatortron_config.lora_alpha,
    lora_dropout=gatortron_config.lora_dropout,
    lora_target_modules=lora_targets,
)

total_params = sum(p.numel() for p in gatortron_model.parameters())
trainable_params = sum(p.numel() for p in gatortron_model.parameters() if p.requires_grad)
print(f"\n=== GatorTron LoRA Model ===")
print(f"  Total parameters:     {total_params:>12,}")
print(f"  Trainable parameters: {trainable_params:>12,}")
print(f"  Trainable fraction:   {100 * trainable_params / total_params:.2f}%")
print(f"  Memory savings:       ~{100 - 100 * trainable_params / total_params:.0f}% fewer trained params")

In [ ]:
# --- Train GatorTron with LoRA on the full ICD dataset ---
import warnings, time
from src.training.trainer import build_trainer
from src.data.preprocessing import preprocess_dataset
from src.data.dataset_loader import get_label_maps

warnings.filterwarnings("ignore", category=FutureWarning)

# Tokenize with GatorTron's tokenizer
gt_l2id, gt_i2l = get_label_maps(icd_label_list)
gt_tok_ds = preprocess_dataset(
    training_dataset, gatortron_tok, gt_l2id,
    max_length=gatortron_config.max_seq_length, num_proc=1,
)

gt_save_dir = "outputs/gatortron_icd_ner_lora/best_model"
gt_trainer = build_trainer(
    model=gatortron_model,
    tokenizer=gatortron_tok,
    train_dataset=gt_tok_ds["train"],
    eval_dataset=gt_tok_ds["test"],
    label_list=icd_label_list,
    training_args_kwargs={
        "output_dir": "outputs/gatortron_icd_ner_lora",
        "num_train_epochs": gatortron_config.num_train_epochs,
        "per_device_train_batch_size": gatortron_config.per_device_train_batch_size,
        "per_device_eval_batch_size": 32,
        "learning_rate": gatortron_config.learning_rate,
        "warmup_ratio": gatortron_config.warmup_ratio,
        "weight_decay": 0.01,
        "fp16": gatortron_config.fp16,
        "logging_steps": 50,
        "eval_strategy": "steps",
        "eval_steps": 200,
        "save_strategy": "steps",
        "save_steps": 200,
        "save_total_limit": 1,
        "load_best_model_at_end": True,
        "metric_for_best_model": "f1",
        "greater_is_better": True,
    },
)

print("Training GatorTron with LoRA adapters...")
t0 = time.time()
gt_train_result = gt_trainer.train()
gt_elapsed = time.time() - t0

gt_metrics = gt_trainer.evaluate()
gt_trainer.save_model(gt_save_dir)
gatortron_tok.save_pretrained(gt_save_dir)

print(f"\n=== GatorTron LoRA Results (Held-Out Test Set) ===")
print(f"  Test F1:    {gt_metrics.get('eval_f1', 0):.4f}")
print(f"  Precision:  {gt_metrics.get('eval_precision', 0):.4f}")
print(f"  Recall:     {gt_metrics.get('eval_recall', 0):.4f}")
print(f"  Eval loss:  {gt_metrics.get('eval_loss', 0):.4f}")
print(f"  Time:       {gt_elapsed:.1f}s")
print(f"  Saved to:   {gt_save_dir}")

# Compare with best 110M model
print(f"\n=== Comparison: GatorTron LoRA vs Best 110M Model ===")
print(f"  {'Model':<25s} {'Params':>12s} {'Trainable':>12s} {'Test F1':>8s}")
print(f"  {'-'*60}")
print(f"  {'GatorTron (LoRA)':<25s} {total_params:>11,} {trainable_params:>11,} "
      f"{gt_metrics.get('eval_f1', 0):>7.4f}")
print(f"  {best_model + ' (full FT)':<25s} "
      f"{icd_results[best_model]['n_params']:>11,} "
      f"{icd_results[best_model]['n_params']:>11,} "
      f"{icd_results[best_model]['f1']:>7.4f}")

In [ ]:
# --- CLI commands for GatorTron LoRA/QLoRA training ---
print("=== GatorTron Training Commands ===")
print()
print("# LoRA (recommended — works on CPU or GPU):")
print("python scripts/train.py --model gatortron-base --dataset icd_ner --lora \\")
print("    --lora-r 16 --lora-alpha 16 --lr 1e-3 --epochs 10")
print()
print("# QLoRA (4-bit — requires CUDA GPU + bitsandbytes):")
print("python scripts/train.py --model gatortron-base --dataset icd_ner --qlora \\")
print("    --lora-r 16 --lora-alpha 16 --lr 1e-3 --epochs 10")
print()
print("# Full fine-tuning (needs more memory — reduce batch size if OOM):")
print("python scripts/train.py --model gatortron-base --dataset icd_ner \\")
print("    --batch-size 8 --grad-accum 2 --epochs 20")

---
## 20. CLI Scripts Reference

Quick reference for the training, prediction, evaluation, and benchmark scripts.

In [ ]:
cli_reference = """
=== Training ===
  python scripts/train.py --model pubmedbert --dataset icd_ner
  python scripts/train.py --model pubmedbert --dataset icd_ner --adversarial
  python scripts/train.py --model pubmedbert --dataset icd_ner --adversarial --adv-method pgd
  python scripts/train.py --model bio_clinicalbert --dataset icd_ner --use-crf --lr 3e-5

=== GatorTron LoRA / QLoRA ===
  python scripts/train.py --model gatortron-base --dataset icd_ner --lora --lr 1e-3
  python scripts/train.py --model gatortron-base --dataset icd_ner --qlora --lr 1e-3
  python scripts/train.py --model gatortron-base --dataset icd_ner --lora --lora-r 32 --lora-alpha 32

=== Prediction ===
  python scripts/predict.py --model-path outputs/pubmedbert_icd_ner/best_model --text "Pt denies cp."
  python scripts/predict.py --model-path outputs/pubmedbert_icd_ner/best_model --icd-codes
  python scripts/predict.py --model-path outputs/pubmedbert_icd_ner/best_model --input-file data.txt --output-file out.json

=== Evaluation ===
  python scripts/evaluate.py --model-path outputs/pubmedbert_icd_ner/best_model --dataset icd_ner
  python scripts/evaluate.py --model-path outputs/pubmedbert_icd_ner/best_model --dataset icd_ner --error-analysis

=== Benchmark ===
  python scripts/benchmark.py --models pubmedbert biobert bio_clinicalbert --datasets icd_ner ncbi_disease bc5cdr

=== Tests ===
  python -m pytest tests/ -v
  python -m pytest tests/ --cov=src --cov-report=term-missing
"""
print(cli_reference)

---
## 21. Running the Test Suite

All tests use mocked models and fallback data — no GPU or network access required.

In [ ]:
# Uncomment to run the full test suite from this notebook:
# !cd {REPO_ROOT} && python -m pytest tests/ -v --tb=short 2>&1 | tail -30

print("To run tests from the command line:")
print(f"  cd {REPO_ROOT}")
print("  python -m pytest tests/ -v")
print()
print("Key test files:")
test_files = [
    ("test_negation.py",            "Rule-based negation detection (200+ assertions)"),
    ("test_assertion.py",           "Transformer assertion classifier"),
    ("test_icd_ner_dataset.py",     "Composite dataset loading + garbage label cleaning"),
    ("test_pipeline.py",            "End-to-end pipeline integration"),
    ("test_icd_pipeline.py",        "ICD code resolution pipeline"),
    ("test_shorthand.py",           "Abbreviation expansion"),
    ("test_disambiguation.py",      "Abbreviation disambiguation"),
    ("test_preprocessing.py",       "Tokenization and label alignment"),
    ("test_entity_postprocessing.py", "Entity filtering and merging"),
]
for filename, desc in test_files:
    print(f"  {filename:35s} — {desc}")

---
## Summary

This notebook demonstrated the full Medical Code Intelligence pipeline:

```
Clinical Text
  → ShorthandExpander (abbreviation expansion with offset tracking)
  → NER Model (transformer token classification, BIO scheme)
  → post_process_entities() (stopword filter, fragment merging)
  → NegationDetector or AssertionClassifier (6 assertion statuses)
  → ICDCodeLookup (TF-IDF matching against 51K codes)
  → DRGCostEstimator (ICD-10 → MS-DRG → cost estimate)
  → MedicalEntity list (text, label, negation, ICD codes, DRG info)
```

Every component has offline fallback data so this notebook runs on CPU without network access.